# API Development (FastAPI)

In [1]:
import sys
import threading
import time
from pathlib import Path

import httpx
import nest_asyncio
import pandas as pd
import uvicorn
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

nest_asyncio.apply()

from src.api.main import create_app
from src.api.schemas import ScoreRequest


In [2]:
API_HOST = '127.0.0.1'
API_PORT = 8765
API_BASE = f'http://{API_HOST}:{API_PORT}'

app = create_app()
print('Scoring model:', app.container.scoring_service().model_name())


Scoring model: pipeline-lightgbm-api-signals


### Start the Server

In [3]:
def _start_server():
    uvicorn.run(app, host=API_HOST, port=API_PORT, log_level='error')

server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

for _ in range(20):
    try:
        r = httpx.get(f'{API_BASE}/health', timeout=2)
        if r.status_code == 200:
            print(f'Server is up: {API_BASE}')
            print('Health:', r.json())
            break
    except Exception:
        time.sleep(0.5)
else:
    print('Server did not start.')


Server is up: http://127.0.0.1:8765
Health: {'status': 'ok', 'rules_loaded': 10, 'kb_chunks': 63, 'model': 'pipeline-lightgbm-api-signals'}


### Test — score

In [4]:
score_payloads = [
    {
        'transaction_id': 'TXN-001',
        'TransactionAmt': 42.50,
        'hour': 14,
        'account_age_days': 220,
        'num_txn_last_1h': 1,
        'P_emaildomain': 'gmail.com',
        'card_type': 'debit',
        'country_mismatch': 0,
    },
    {
        'transaction_id': 'TXN-002',
        'TransactionAmt': 875.00,
        'hour': 2,
        'account_age_days': 1,
        'num_txn_last_1h': 9,
        'P_emaildomain': 'protonmail.com',
        'card_type': 'credit',
        'country_mismatch': 1,
    },
]

score_rows = []
for payload in score_payloads:
    r = httpx.post(f'{API_BASE}/score', json=payload)
    score_rows.append(r.json())
    print(f'POST /score  →  {r.status_code}  {r.json()}')

display(pd.DataFrame(score_rows))


POST /score  →  200  {'transaction_id': 'TXN-001', 'anomaly_score': 0.4862, 'risk_level': 'MEDIUM', 'triggered_signals': []}
POST /score  →  200  {'transaction_id': 'TXN-002', 'anomaly_score': 0.657, 'risk_level': 'HIGH', 'triggered_signals': ['is_night_transaction', 'is_new_account', 'is_high_amount', 'has_velocity_spike', 'uses_disposable_email', 'uses_credit_card', 'country_mismatch']}


,transaction_id,anomaly_score,risk_level,triggered_signals
0,TXN-001,0.4862,MEDIUM,[]
1,TXN-002,0.6570,HIGH,"[is_night_transaction, is_new_account, is_high..."


### Test — explain

In [6]:
explain_payload = score_payloads[1]
r = httpx.post(f'{API_BASE}/explain', json=explain_payload)
result = r.json()
print(f"transaction_id : {result['transaction_id']}")
print(f"anomaly_score  : {result['anomaly_score']}")
print(f"risk_level     : {result['risk_level']}")
print(f"explanation    :\n  {result['explanation']}")


transaction_id : TXN-002
anomaly_score  : 0.657
risk_level     : HIGH
explanation    :
  This transaction has a risk score of 0.657, which is considered HIGH risk. It was flagged because of the following: is night transaction, is new account, is high amount, has velocity spike, uses disposable email, uses credit card, country mismatch. The transaction was for 875.00 at 02:00, from an account that is 1 day(s) old.


### Test — rules/evaluate

In [8]:
rules_payload = {
    'transaction_id': 'TXN-003',
    'features': {
        'adjusted_anomaly_score': 0.82,
        'anomaly_high_flag_count': 11,
        'is_high_risk_entity': 1,
        'final_context_anomaly_score': 0.77,
        'is_outlier': 1,
        'ctx_amount_zscore': 4.5,
        'TransactionAmt': 310.0,
        'is_business_hour': 0,
        'anomaly_high_flag_ratio': 0.40,
        'time_since_first_transaction': 3600,
        'is_night_transaction': 1,
        'entity_trusted_score': 0.22,
        'ctx_amount_vs_product_avg': 350,
        'ctx_amount_ratio_to_global_median': 6.2,
    },
}

r = httpx.post(f'{API_BASE}/rules/evaluate', json=rules_payload)
result = r.json()
print(f"final_action   : {result['final_action']}")
print(f"final_severity : {result['final_severity']}")
print(f"fired_rules    : {len(result['fired_rules'])}")
display(pd.DataFrame(result['fired_rules'])[['rule_id', 'name', 'severity', 'action']])


final_action   : FLAG_CRITICAL_REVIEW
final_severity : CRITICAL
fired_rules    : 10


,rule_id,name,severity,action
0,RULE_001,Extreme Composite Anomaly Risk,CRITICAL,FLAG_CRITICAL_REVIEW
1,RULE_002,High-Risk Entity with Strong Context Risk,CRITICAL,FLAG_CRITICAL_REVIEW
2,RULE_003,Outlier with Very Strong Model Risk,CRITICAL,FLAG_CRITICAL_REVIEW
3,RULE_004,Extreme Amount Deviation,CRITICAL,FLAG_CRITICAL_REVIEW
4,RULE_005,Off-Hours Strong Fraud Signal,HIGH,FLAG_REVIEW
5,RULE_006,Product Amount Spike with High Risk,HIGH,FLAG_REVIEW
6,RULE_007,High Global Amount Ratio with High Risk,HIGH,FLAG_REVIEW
7,RULE_008,Dense Anomaly Pattern,HIGH,FLAG_REVIEW
8,RULE_009,New Account Velocity Risk,HIGH,FLAG_REVIEW
9,RULE_010,Night Transaction with Untrusted Entity,MEDIUM,FLAG_REVIEW


### Test — rules/list

In [10]:
r = httpx.get(f'{API_BASE}/rules/list')
catalog = r.json()
display(pd.DataFrame(catalog['rules'])[['rule_id', 'name', 'severity', 'action']].head())


,rule_id,name,severity,action
0,RULE_001,Extreme Composite Anomaly Risk,CRITICAL,FLAG_CRITICAL_REVIEW
1,RULE_002,High-Risk Entity with Strong Context Risk,CRITICAL,FLAG_CRITICAL_REVIEW
2,RULE_003,Outlier with Very Strong Model Risk,CRITICAL,FLAG_CRITICAL_REVIEW
3,RULE_004,Extreme Amount Deviation,CRITICAL,FLAG_CRITICAL_REVIEW
4,RULE_005,Off-Hours Strong Fraud Signal,HIGH,FLAG_REVIEW


### Test — rag/query

In [11]:
rag_queries = [
    'What happens when a new account makes a high-value transaction?',
    'What happens when an old account makes a high-value transaction?',
    'How is conflict resolution handled when multiple rules fire?',
]

for query in rag_queries:
    r = httpx.post(f'{API_BASE}/rag/query', json={'query': query, 'top_k': 1})
    result = r.json()
    print(f"\nQuery : {query}")
    print(f"Status: {r.status_code}  |  Results: {len(result['results'])}")
    for chunk in result['results']:
        print(f"  score={chunk['score']:.4f}  source={chunk['source']}")
        print(f"    {chunk['text'][:120].replace(chr(10), ' ')}...")



Query : What happens when a new account makes a high-value transaction?
Status: 200  |  Results: 1
  score=3.0443  source=fraud_policy.md
    ### 7. Established Account High-Value Transactions When an old, long-standing, or established account with a positive hi...

Query : What happens when an old account makes a high-value transaction?
Status: 200  |  Results: 1
  score=3.0443  source=fraud_policy.md
    ### 7. Established Account High-Value Transactions When an old, long-standing, or established account with a positive hi...

Query : How is conflict resolution handled when multiple rules fire?
Status: 200  |  Results: 1
  score=1.3653  source=fraud_policy.md
    ## Conflict Resolution Policy  When multiple fraud rules fire for the same transaction: 1. The rule with the highest pri...
